# Validate one daily load
Run blocking source-to-Bronze-to-Silver-to-Gold controls before promotion or completion.

In [ ]:
from pathlib import Path
import sys

source_root = next((root / "src" for root in (Path.cwd(), *Path.cwd().parents) if (root / "src").is_dir()), None)
if source_root is not None and str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

In [ ]:
ENVIRONMENT = "dev"
SOURCE_URI = ""
try:
    dbutils.widgets.text("environment", ENVIRONMENT)
    dbutils.widgets.text("source_uri", SOURCE_URI)
    ENVIRONMENT = dbutils.widgets.get("environment").strip().lower()
    SOURCE_URI = dbutils.widgets.get("source_uri").strip()
except NameError:
    pass
if ENVIRONMENT not in {"dev", "prod"}:
    raise ValueError("environment must be dev or prod")
if not SOURCE_URI:
    raise ValueError("Set source_uri to the daily Parquet file to validate.")

In [ ]:
from finops_cloud.audit.daily_controls import validate_daily_load
from finops_cloud.config import load_config
from finops_cloud.runtime import get_spark

config = load_config(ENVIRONMENT)
spark_session = get_spark(config.profile)
result = validate_daily_load(spark_session, config, SOURCE_URI)
display(spark_session.createDataFrame([result]))
try:
    dbutils.jobs.taskValues.set(key="validation_status", value=result["status"])
    dbutils.jobs.taskValues.set(key="billing_month", value=result["billing_month"])
except NameError:
    pass